# Rekurencyjne Sieci Neuronowe (RNN)

* 24.11.25 Jadwiga Krząstek

Zagadnienia na dziś:
- Rekurencyjne sieci neuronowe
- LSTM
- Reprezentacje słów: One-hot encoding, Embeddingi 

<h4> Zadanie1 (2pkt, RNN/LSTM vs klasyczna reprezentacja)

- Rozważ plik all_chem_df.csv, zawierający pewne leki (w formacie SMILES) i informację o obszarze działania. Wybierz 4 najczęstsze targety. W oparciu o reprezentację One-hot-encoding oraz sieci rekurencyjne zbuduj klasyfikator. Wydziel zbiór treningowy i testowy (a być może także walidacyjny).
- W pliku drugs_prop.txt występują te same leki, ale tym razem podano ich wybrane własności fizykochemiczne. Zbuduj klasyfikator w oparciu o regresję logistyczną lub/oraz SVM. Wydziel zbiór treningowy i testowy.
- Porównaj efektywność zbudowanych modeli. Skomentuj otrzymane wyniki. Wskaż plusy i minusy obydwu rozwiązań.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
import pandas as pd

df = pd.read_csv("all_chem_df.csv", sep = ",")

from collections import Counter
print(Counter(df["tags"]).most_common(4))

df2 = df[df['tags'].isin(['antiinfective', 'antineoplastic', 'cns','cardio'])]
print(df2.head())

X = list(df2["smiles"])
zlaczone = "".join(list(df2["smiles"]))

[('antiinfective', 2412), ('antineoplastic', 1175), ('cns', 1149), ('cardio', 797)]
  image_name            tags  \
1     pics/1   antiinfective   
2     pics/2   antiinfective   
3     pics/3  antineoplastic   
5     pics/5             cns   
6     pics/6             cns   

                                              smiles                Col3  
1  CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(N...   ['antiinfective']  
2  CCCCC(C)C(=O)OC1C(C)C(CC)OC2(CC3CC(C/C=C(\C)CC...   ['antiinfective']  
3  COc1cc2c(c(OC)c1OC)-c1c(cc3c(c1OC)OCO3)C[C@H](...  ['antineoplastic']  
5  CC(=O)OCC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C             ['cns']  
6                         CC(=O)Nc1nnc(S(N)(=O)=O)s1             ['cns']  


In [ ]:
print(X[0])

CCC[C@@]1(CCc2ccccc2)CC(O)=C([C@H](CC)c2cccc(NS(=O)(=O)c3ccc(C(F)(F)F)cn3)c2)C(=O)O1


#### Alfabet i SMILES
Sprawdzimy jakie znaki wytępują w formacie i wykorzystamy je do stworzenia alfabetu.

In [ ]:
chars = set("".join(df2["smiles"].astype(str)))
alphabet = sorted(chars)
char_to_idx = {c: i for i, c in enumerate(alphabet)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

n_letters = len(alphabet)
print("Alphabet size:", n_letters)
print("Alphabet:", alphabet)

Alphabet size: 53
Alphabet: ['#', '%', '(', ')', '+', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'A', 'B', 'C', 'F', 'G', 'H', 'I', 'M', 'N', 'O', 'P', 'S', 'T', 'X', 'Z', '[', '\\', ']', 'a', 'b', 'c', 'd', 'e', 'g', 'i', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u']


In [ ]:
def text_to_onehot(text):
    seq = torch.zeros(len(text), n_letters)
    for i, ch in enumerate(text):
        seq[i, char_to_idx[ch]] = 1.0
    return seq

X = list(df2["smiles"])
y = df2["tags"].astype("category").cat.codes

#60% train, 20% val, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

#konieczna zamiana na listy (problem z indeksami)
X_train = list(X_train)
X_val   = list(X_val)
X_test  = list(X_test)

y_train = list(y_train)
y_val   = list(y_val)
y_test  = list(y_test)

#Dataset, collate_fn — jak wyżej
class ListDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return text_to_onehot(self.data[idx]), torch.tensor(self.targets[idx], dtype=torch.long)

def collate_fn(batch):
    #batch = [(seq1, label1), (seq2, label2), ...]
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True)  #wymiar: (batch, max_len, vocab_size)
    labels = torch.stack(labels)
    return padded, labels, lengths



In [ ]:
train_targets = list(y_train)
uni, counts = np.unique(train_targets, return_counts=True)
weights = len(train_targets) / counts
weights_per_sample = [weights[t] for t in train_targets]

sampler = WeightedRandomSampler(weights_per_sample, num_samples=len(weights_per_sample))

train_dataset = ListDataset(X_train, y_train)
val_dataset   = ListDataset(X_val, y_val)
test_dataset  = ListDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)


In [ ]:
class LSTMClassifier(nn.Module): #model jak wyżej
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (hidden, cell) = self.lstm(packed)
        out = self.dropout(hidden[-1])
        out = self.fc(out)
        return out

#Budujemy model
num_classes = len(set(y))
model = LSTMClassifier(input_size=n_letters, hidden_size=16, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

#Trenujemy model
for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch, lengths in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch, lengths)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss / len(train_loader):.4f}")

#Ocena modelu
def evaluate(model, loader, topk=(1,3)):
    model.eval()
    correct = {k: 0 for k in topk}
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            outputs = model(X_batch, lengths)
            total += y_batch.size(0)
            for k in topk:
                _, pred = outputs.topk(k, dim=1)
                correct[k] += (pred == y_batch.view(-1, 1)).any(dim=1).sum().item()

    for k in topk:
        acc = 100 * correct[k] / total
        print(f"Top-{k} accuracy: {acc:.2f}%")

evaluate(model, test_loader)


Epoch 01 | Loss: 1.3628
Epoch 02 | Loss: 1.3503
Epoch 03 | Loss: 1.3356
Epoch 04 | Loss: 1.2839
Epoch 05 | Loss: 1.2552
Epoch 06 | Loss: 1.1922
Epoch 07 | Loss: 1.1480
Epoch 08 | Loss: 1.1210
Epoch 09 | Loss: 1.1000
Epoch 10 | Loss: 1.0542
Top-1 accuracy: 55.19%
Top-3 accuracy: 91.33%


#### Część 2: regresja logistyczna

In [ ]:
df_prop = pd.read_csv("drugs_prop.txt", sep=",") 

X = df_prop[["mass", " logp", " h_d", " h_a", " rot_b", " tpsa"]]
y = df_prop[" target"].astype("category").cat.codes

#train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(
    solver="lbfgs",
    max_iter=50000,
    class_weight="balanced"
)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)

print(classification_report(y_test, y_pred_lr))


              precision    recall  f1-score   support

           0       0.70      0.47      0.56       965
           1       0.31      0.27      0.28       470
           2       0.30      0.33      0.31       319
           3       0.43      0.76      0.55       460

    accuracy                           0.46      2214
   macro avg       0.43      0.45      0.43      2214
weighted avg       0.50      0.46      0.46      2214



#### Część 3: SVM

Klasy dość niezbalansowane, ale SMOTE nie pomogło, musi wystarczyć class_weight="balanced"

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

df_prop = pd.read_csv("drugs_prop.txt", sep=",") 

X = df_prop[["mass", " logp", " h_d", " h_a", " rot_b", " tpsa"]]
y = df_prop[" target"].astype("category").cat.codes

#train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

#pipeline: standaryzacja + SVM
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=20, gamma=0.1, class_weight="balanced"))
])

clf.fit(X_train, y_train) #trening

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.80      0.49      0.61       603
           1       0.43      0.48      0.45       294
           2       0.39      0.51      0.44       199
           3       0.51      0.76      0.61       288

    accuracy                           0.54      1384
   macro avg       0.53      0.56      0.53      1384
weighted avg       0.60      0.54      0.55      1384



### **Komentarz**

#### Podsumowanie jakości modeli

- **LSTM wypada najlepiej**, zwłaszcza przy Top-3 accuracy – model rekurencyjny, uwzględnia kontekst
- **Regresja logistyczna**  jest stabilna, prosta i szybka, ale ma ograniczoną zdolność uchwycenia złożonych wzorców
- **SVM** poradził sobie najgorzej, może to wynikać z ograniczeń przy danych tekstowych, które są bardziej złożone i nieliniowe.

#### Plusy i minusy modeli

| Model               |Wynik | Plusy | Minusy |
|---------------------|----|-------|--------|
| **LSTM** | **55%**   | – dał najlepsze wyniki<br>– uchwycił zależności sekwencyjne<br>– dobre wyniki Top-3 accuracy | – najbardziej złożony obliczeniowo<br>– wymaga dużo danych i dostrajania hiperparametrów |
| **SVM** | **54%**   |– dobrze działa w problemach o niskiej liczbie cech| – słaba skuteczność na danych tekstowych<br>– dostrajanie hiperparametrów |
| **Regresja logistyczna** | **46%**   |– szybka<br>– prosta i łatwa w interpretacji<br> | – model liniowy – ograniczona zdolność modelowania złożonych danych<br>– mniejsza elastyczność niż SVM i LSTM |


<h4> Zadanie2 (Jak zakodować aminokwasy? Embeddingi)
    
- Rozważ dane dotyczącece lokalizacji komórkowej wybranych białek (peptydów) - te same co w ćw2 (z regresji logistycznej). Tym razem, zbuduj model w oparciu o sieci rekurencyjne i zaproponowaną przez siebie reprezentację aminokwasów - tzw. embeddingi. Potestuj różne topologie sieci oraz reprezentacje dla danych. Skomentuj otrzymane wyniki

#### Próba 1: ta sama reprezentacja jak w ćwiczeniach 2, tylko model to LSTM.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from collections import Counter
from Bio import SeqIO

In [ ]:
#WCZYTANIE I PRZYGOTOWANIE DANYCH, WYBÓR REPREZENTACJI, STANDARYZACJA, ZBILANSOWANIE
df = pd.read_csv("swissprot_annotated_proteins.tab", sep="\t", header=None)
df.columns = ["protein_id", "localization", "feature"]
print(df.head())

seq_data = []
for record in SeqIO.parse("targetp.fasta", "fasta"):
    seq_data.append({"protein_id": record.id, "sequence": str(record.seq)})

seq_df = pd.DataFrame(seq_data)
print(seq_df.head())

#łaczenie danych
merged_df = pd.merge(df, seq_df, on="protein_id", how="inner")
print(f"\nPołączono dane: {merged_df.shape[0]} rekordów")
print(merged_df.head())

merged_df = merged_df[merged_df["localization"] != "Other"].copy()

print("\nLiczność po odrzuceniu 'Other':")
print(merged_df["localization"].value_counts())

#reprezentacja sekwencji - obliczanie zawartości procentowej aminokwasów danej grupy w całej sekwencji
def seq_features(seq):
    seq = seq.upper()
    length = len(seq)
    hydrophobic = sum(aa in "AILMFWYV" for aa in seq) / length
    acidic = sum(aa in "DE" for aa in seq) / length
    basic = sum(aa in "KRH" for aa in seq) / length
    polar = sum(aa in "STNQ" for aa in seq) / length
    return pd.Series([length, hydrophobic, acidic, basic, polar])

merged_df[["length", "hydrophobic", "acidic", "basic", "polar"]] = merged_df["sequence"].apply(seq_features)
X = merged_df[["length", "hydrophobic", "acidic", "basic", "polar"]].values
y = merged_df["localization"].values

#standaryzacja ===
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

#SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

balanced_df = pd.DataFrame(X_res, columns=["length", "hydrophobic", "acidic", "basic", "polar"])
balanced_df["localization"] = y_res

print("\nLiczności po zbilansowaniu:")
print(balanced_df["localization"].value_counts())

  protein_id localization  feature
0     P10719           MT       46
1     Q38786           CH       55
2     P15289           SP       18
3     P25705           MT       43
4     P00829           MT       48
  protein_id                                           sequence
0     P92192  MKFLIVFVALFAMAVARPNLAEIVRQVSDVEPEKWSSDVETSDGTS...
1     P30042  MAAVRALVASRLAAASAFTSLSPGGRTPSQRAALHLSVPRPAARVA...
2     Q9D666  MDFSRLHTYTPPQCVPENTGYTYALSSSYSSDALDFETEHKLEPVF...
3     Q9BUL9  MENFRKVRSEEAPAGCGAEGGGPGSGPFADLAPGAVHMRVKEGSKI...
4     U6A629  MLAEYLLLPLLASYASAVTISVAKSGGNVTTGLQYGAMEEEINHCG...

Połączono dane: 13005 rekordów
  protein_id localization  feature  \
0     P10719           MT       46   
1     Q38786           CH       55   
2     P15289           SP       18   
3     P25705           MT       43   
4     P00829           MT       48   

                                            sequence  
0  MLSLVGRVASASASGALRGLNPLAALPQAHLLLRTAPAGVHPARDY...  
1  MALLCSALSNSTHPSFRSHIGANSENLWHLSA

In [ ]:
#train/test (80/20), train/val (60/20/20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

#standaryzacja danych tylko na zbiorze treningowym
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (hidden, cell) = self.lstm(packed)
        out = self.dropout(hidden[-1])  # ostatnia warstwa, ostatni stan
        out = self.fc(out)
        return out

#Budujemy model
num_classes = len(set(y))
model = LSTMClassifier(input_size=n_letters, hidden_size=16, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

#Trenujemy model
for epoch in range(5):
    model.train()
    total_loss = 0
    for X_batch, y_batch, lengths in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch, lengths)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss / len(train_loader):.4f}")

#Ocena modelu
def evaluate(model, loader, topk=(1,3)):
    model.eval()
    correct = {k: 0 for k in topk}
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            outputs = model(X_batch, lengths)
            total += y_batch.size(0)
            for k in topk:
                _, pred = outputs.topk(k, dim=1)
                correct[k] += (pred == y_batch.view(-1, 1)).any(dim=1).sum().item()

    for k in topk:
        acc = 100 * correct[k] / total
        print(f"Top-{k} accuracy: {acc:.2f}%")

evaluate(model, test_loader)

Epoch 01 | Loss: 1.3408
Epoch 02 | Loss: 1.3167
Epoch 03 | Loss: 1.3231
Epoch 04 | Loss: 1.2916
Epoch 05 | Loss: 1.2443
Top-1 accuracy: 35.95%
Top-3 accuracy: 69.02%


#### Wyniki nie są zadowalające, dodajmy teraz kilka ulepszeń:
- więcej warstw
- dropout - lepsza generalizacja, mniej overfittingu
- batchNorm - stabilniejszy trening
- większe hidden_size

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.bn = nn.BatchNorm1d(hidden_size * 2)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, (hidden, cell) = self.lstm(packed)

        #hidden shape: (num_layers * 2, batch, hidden_size)
        #bierzemy ostatnią warstwę: [fw, bw]
        fw = hidden[-2]
        bw = hidden[-1]

        out = torch.cat((fw, bw), dim=1)

        out = self.dropout(out)
        out = self.bn(out)
        out = self.fc(out)

        return out

num_classes = len(set(y))
model = LSTMClassifier(input_size=n_letters, hidden_size=16, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch, lengths in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch, lengths)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss / len(train_loader):.4f}")

evaluate(model, test_loader)

Epoch 01 | Loss: 1.3944
Epoch 02 | Loss: 1.3658
Epoch 03 | Loss: 1.3431
Epoch 04 | Loss: 1.3376
Epoch 05 | Loss: 1.3095
Epoch 06 | Loss: 1.2612
Epoch 07 | Loss: 1.2226
Epoch 08 | Loss: 1.1867
Epoch 09 | Loss: 1.1820
Epoch 10 | Loss: 1.1455
Top-1 accuracy: 52.21%
Top-3 accuracy: 89.34%


#### W drugiej próbie uzyskaliśmy lepsze wyniki dla tej reprezentacji. Ten sposób jednak nie ma dużego sensu biologicznego, bo wektor liczb nie ma własności sensownego zdania/sekwencji i konieczności zapamiętywania poprzednich informacji.
Teraz wykorzystamy embeddingi i model BiLSTM, który uzyskał najlepsze wartości accuracy ze wszystkich testowanych. BiLSTM jest modelem, który przechodzi po danych od lewej i od prawej strony.

In [ ]:
from sklearn.model_selection import train_test_split
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd
from Bio import SeqIO
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


In [ ]:
df = pd.read_csv("swissprot_annotated_proteins.tab", sep="\t", header=None)
df.columns = ["protein_id", "localization", "feature"]

#wczytanie sekwencji
seq_data = []
for record in SeqIO.parse("targetp.fasta", "fasta"):
    seq_data.append({"protein_id": record.id, "sequence": str(record.seq)})

seq_df = pd.DataFrame(seq_data)

#połączenie
merged_df = pd.merge(df, seq_df, on="protein_id", how="inner")

#usuwamy "Other"
merged_df = merged_df[merged_df["localization"] != "Other"].copy()
merged_df.reset_index(drop=True, inplace=True)

print("Liczność klas:")
print(merged_df["localization"].value_counts())

#tworzenie alfabetu
alphabet = sorted({aa for seq in merged_df["sequence"] for aa in seq})
print("Alfabet:", alphabet)

#słownik
char_to_idx = {c: i+1 for i, c in enumerate(alphabet)}
char_to_idx["PAD"] = 0

idx_to_char = {v: k for k, v in char_to_idx.items()}

vocab_size = len(char_to_idx)
print("Vocab size:", vocab_size)

Liczność klas:
localization
SP    2697
MT     499
CH     227
TH      45
Name: count, dtype: int64
Alfabet: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Vocab size: 25


Utworzenie datasetów i dataloaderów

In [ ]:
def seq_to_indices(seq): #zamiana sekwencji na indeksy
    return torch.tensor([char_to_idx.get(aa, 0) for aa in seq], dtype=torch.long)

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class ProteinSeqDataset(Dataset): #dataset + collate_fn (padding inside batch)
    def __init__(self, df):
        self.seqs = df["sequence"].tolist()
        self.labels = df["localization"].astype("category").cat.codes.values

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        return seq_to_indices(self.seqs[idx]), torch.tensor(self.labels[idx], dtype=torch.long)

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return padded, labels, lengths


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    merged_df, test_size=0.2, stratify=merged_df["localization"], random_state=42
)

train_df, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df["localization"], random_state=42
)

train_ds = ProteinSeqDataset(train_df)
val_ds   = ProteinSeqDataset(val_df)
test_ds  = ProteinSeqDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=32, collate_fn=collate_fn)


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes,
                 num_layers=1, dropout=0.3):
        super().__init__()

        #embeddingi
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        #LSTM
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)  #(batch, seq_len, emb_dim)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        #LSTM -> zwraca hidden (layers * directions, batch, hidden_size)
        packed_out, (hidden, cell) = self.lstm(packed)
        fw = hidden[-2]
        bw = hidden[-1]
        h = torch.cat([fw, bw], dim=1)   # (batch, hidden_size*2)

        h = self.dropout(h)
        out = self.fc(h)
        return out


In [ ]:
char_to_idx = {c: i for i, c in enumerate(alphabet)}

vocab_size = len(char_to_idx)          #np. 21
emb_dim = 32                         #testy 16 / 32 / 64
hidden_size = 64                     #testy32 / 64 / 128
num_classes = len(set(y))            #np. 4

model = BiLSTMClassifier(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    hidden_size=hidden_size,
    num_classes=num_classes,
    num_layers=1,
    dropout=0.3
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

#Trenujemy model
for epoch in range(5):
    model.train()
    total_loss = 0
    for X_batch, y_batch, lengths in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch, lengths)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss / len(train_loader):.4f}")

#Ocena modelu
def evaluate(model, loader, topk=(1,3)):
    model.eval()
    correct = {k: 0 for k in topk}
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            outputs = model(X_batch, lengths)
            total += y_batch.size(0)
            for k in topk:
                _, pred = outputs.topk(k, dim=1)
                correct[k] += (pred == y_batch.view(-1, 1)).any(dim=1).sum().item()

    for k in topk:
        acc = 100 * correct[k] / total
        print(f"Top-{k} accuracy: {acc:.2f}%")

evaluate(model, test_loader)

Epoch 01 | Loss: 0.5792
Epoch 02 | Loss: 0.2427
Epoch 03 | Loss: 0.1405
Epoch 04 | Loss: 0.1280
Epoch 05 | Loss: 0.0991
Top-1 accuracy: 94.24%
Top-3 accuracy: 99.42%


Ostatni wynik LSTM okazał się być znacznie lepszy od poprzednich prób

#### **Komentarz**
W pierwszej części zadania używaliśmy ręcznie wybranych cech (długość sekwencji, pI, masa cząsteczkowa, zawartość aminokwasów), które są mocno uproszczoną reprezentacją sekwencji aminokwasowej. Te cechy tracą informacje o kolejności aminokwasów, które jest bardzo ważna w przypadku analizy sekwencji. \
Zalety LSTM:
- utrzymuje pamięć długoterminową
- rozpoznaje motywy i wzorce w dowolnym miejscu sekwencji
- radzi sobie z różnymi długościami sekwencji

BiLSTM składa się z dwóch LSTM: jednego czytającego sekwencję od lewej, drugiego od prawej. Pozwala jeszcze dokładniej przeanalizować sekwencje i wyłapać zależności pomiędzy sąsiadującymi aminokwasami.
